# Whispers — Phase 2 hybrid self-play (MARS-style)

Replaces the scripted *honest* baselines with the Phase-1 LoRA, while the
*adversaries* remain scripted (so the agent has to keep improving against
a stable adversarial distribution). We adopt two MARS / MARSHAL tricks to
keep multi-turn multi-agent GRPO stable:

1. **Turn-level advantage** — instead of crediting the entire trajectory
   reward to every token, the advantage at turn `t` uses the *suffix*
   reward `R_t = sum_{k>=t} r_k` (γ = 1).
2. **Agent-specific advantage normalisation** — when multiple seats are
   filled by the same LoRA, advantages are z-scored *within each seat*
   before being passed to the loss, so a high-variance witness role does
   not drown out a low-variance editor role.

References: MARS (Reinforcing Multi-Agent Reasoning of LLMs through
Self-Play, 2025); MARSHAL (multi-agent SHAP-aware Loss, 2025).

## 0. Setup (mirrors phase 1)

In [ ]:
import os, sys, subprocess, time, json, math, random
from pathlib import Path
from collections import defaultdict
import numpy as np
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
REPO_DIR = '/content/whispers' if IN_COLAB else '..'
sys.path.insert(0, REPO_DIR)
ASSETS = Path(REPO_DIR) / 'assets'
print('repo:', REPO_DIR)

## 1. Load Phase-1 LoRA checkpoint

In [ ]:
MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'
ADAPTER_DIR = os.environ.get('ADAPTER_DIR', f'{REPO_DIR}/runs/phase1-lora')
model = tokenizer = None
try:
    from unsloth import FastLanguageModel
    model, tokenizer = FastLanguageModel.from_pretrained(
        MODEL_NAME, max_seq_length=2048, load_in_4bit=True, dtype=None,
    )
    if Path(ADAPTER_DIR).exists():
        from peft import PeftModel
        model = PeftModel.from_pretrained(model, ADAPTER_DIR)
        print(f'Loaded Phase-1 LoRA from {ADAPTER_DIR}')
    else:
        print(f'WARNING: {ADAPTER_DIR} not found; starting from base model.')
except Exception as e:
    print(f'Skipping model load: {type(e).__name__}: {e}')

## 2. Multi-seat rollout — protagonist + LoRA-honest fillers + scripted adversaries

We monkey-patch `whispers.sim.scripted_policy` so that *honest* non-protagonist seats query the LoRA instead of using the deterministic heuristic. Adversary seats keep the original scripted lie-injection behaviour.

In [ ]:
from whispers import sim as wsim
from whispers.env import WhispersEnv
from whispers.models import WhispersAction
from inference import _build_user_prompt, _coerce_action, SYSTEM_PROMPT  # type: ignore

_orig_policy = wsim.scripted_policy

def _llm_action_for(role_obs):
    if model is None:
        return None
    prompt = SYSTEM_PROMPT + '\n\n' + _build_user_prompt(role_obs)
    inputs = tokenizer(prompt, return_tensors='pt').to(model.device)
    out_ids = model.generate(**inputs, max_new_tokens=128, do_sample=True, temperature=0.6, top_p=0.9)
    raw = tokenizer.decode(out_ids[0][inputs.input_ids.shape[-1]:], skip_special_tokens=True)
    action, _err = _coerce_action(raw, role_obs)
    return action

def hybrid_policy(ctx):
    if ctx.agent.role == 'adversary':
        return _orig_policy(ctx)
    # Build a synthetic per-seat observation so the LLM sees the same shape it was trained on
    from whispers.tools import legal_tools_for
    role_obs = ctx.state.agents[ctx.agent.agent_id]
    from whispers.models import WhispersObservation
    obs = WhispersObservation(
        role=role_obs.role, agent_id=role_obs.agent_id,
        inbox=list(role_obs.inbox), public_feed=ctx.state.public_feed[-12:],
        private_facts=list(role_obs.private_facts),
        network_neighbors=ctx.state.network.get(role_obs.agent_id, []),
        fact_check_budget=role_obs.fact_check_budget,
        step=ctx.state.step, max_steps=ctx.state.max_steps,
        legal_tools=legal_tools_for(ctx.state),
        task_id=ctx.state.task_id,
        objective=f'You are filler agent {role_obs.agent_id} (role={role_obs.role}).',
    )
    a = _llm_action_for(obs)
    return a if a is not None else _orig_policy(ctx)

wsim.scripted_policy = hybrid_policy
print('hybrid_policy installed:', wsim.scripted_policy.__name__)

## 3. MARS-style turn-level advantage + agent-specific normalisation

We compute advantages from per-step rewards rather than from a single terminal score, then z-score within each (role, task) bucket.

In [ ]:
def turn_level_advantages(per_step_rewards: list[float]) -> list[float]:
    """R_t = sum_{k >= t} r_k (gamma=1). Returns one scalar per turn."""
    R = 0.0
    out = [0.0] * len(per_step_rewards)
    for t in range(len(per_step_rewards) - 1, -1, -1):
        R = per_step_rewards[t] + R
        out[t] = R
    return out

def agent_specific_normalise(buckets: dict[str, list[float]]) -> dict[str, list[float]]:
    out = {}
    for k, vals in buckets.items():
        if len(vals) < 2:
            out[k] = [0.0 for _ in vals]
            continue
        mu = sum(vals) / len(vals)
        sd = math.sqrt(sum((v - mu) ** 2 for v in vals) / max(1, len(vals) - 1)) or 1.0
        out[k] = [(v - mu) / sd for v in vals]
    return out

## 4. Phase 2 outer loop (100-200 steps; 6 generations per step)

In [ ]:
PHASE2_STEPS = 150
GEN_PER_STEP = 6
TASKS = ['t3', 't4', 't5', 't5', 't6']

phase2_history = {'step': [], 'task_id': [], 'score': [], 'adv_normed_var': []}
for i in range(PHASE2_STEPS):
    tid = TASKS[i % len(TASKS)]
    seed = 9000 + i
    scores, per_step_buckets = [], defaultdict(list)
    for g in range(GEN_PER_STEP):
        env = WhispersEnv(task_id=tid, seed=seed + g)
        obs = env.reset()
        rs = []
        for _ in range(min(18, obs.max_steps)):
            # Reuse phase 1 policy: random or LLM
            tool = random.choice([t for t in obs.legal_tools if t != 'fact_check'])
            obs, r, done, _info = env.step(WhispersAction(tool=tool))
            rs.append(float(r.value))
            if done:
                break
        adv = turn_level_advantages(rs)
        per_step_buckets[(env.state.agents[env.state.protagonist_id].role, tid)].extend(adv)
        scores.append(float(env.grade_terminal()['value']))
    normed = agent_specific_normalise(per_step_buckets)
    var = float(np.var([v for vs in normed.values() for v in vs])) if normed else 0.0
    phase2_history['step'].append(i)
    phase2_history['task_id'].append(tid)
    phase2_history['score'].append(float(np.mean(scores)))
    phase2_history['adv_normed_var'].append(var)
    if i % 25 == 0:
        print(f'phase2 step {i:3d}  task={tid}  score={phase2_history["score"][-1]:.3f}  adv_var={var:.3f}')

(ASSETS / 'phase2_history.json').write_text(json.dumps(phase2_history))
print('saved', ASSETS / 'phase2_history.json')

## 5. Plot phase-2 progression

In [ ]:
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(phase2_history['step'], phase2_history['score'], label='self-play score')
ax.set_xlabel('Phase-2 step'); ax.set_ylabel('Mean episode score (in [0,1])')
ax.set_title('Phase 2: hybrid self-play (MARS-style)')
ax.grid(True, alpha=0.3); ax.legend()
fig.tight_layout(); fig.savefig(ASSETS / 'phase2_curve.png', dpi=150)
print('saved', ASSETS / 'phase2_curve.png')